In [1]:
# ruff: noqa: F401

import os
from pathlib import Path

import board
from dotenv import load_dotenv
from fastmcp.client.transports import StdioTransport
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from rich.console import Console
from rich.markdown import Markdown

load_dotenv(override=True)
console = Console()

### Create Agent (Using OpenAI Chat Completion API)


In [2]:
MODEL = "openai-chat:gpt-5.4-nano"

agent = Agent(model=MODEL, system_prompt="You are a concise frendly assistant. Answer in a single short sentence")

### Run the agent


In [3]:
result = await agent.run(user_prompt="Say hello in Bengali")
print(result.output)

                 pydantic-ai v2.50.0 • Python 3.14.7
      / \
     /   \       agent: agent • model: openai:gpt-5.4-nano • tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


নমস্কার!


### Using SQLite Todo board


In [4]:
board.reset_board()
board.add_goal(
    """
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.
    """
)
board.show_board()
board.list_todos()

Goal #1: 
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.

[{'id': 1,
  'parent_id': None,
  'task': '\n    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.\n    Overwrite the file if it already exists.\n    ',
  'status': 'pending',
  'result': ''}]

### Add Tools (@tool decorators not needed in Pydantin AI)


In [5]:
def show_todos() -> list[dict]:
    """List every tool in the todo list. A goal has a parent id is None. A step has the parent_id set to it's goal_id"""
    return board.list_todos()


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps

    Args:
        goal_id: The id of the goal to decompose
        steps: Short description of the steps to be taken in order

    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (step or goal) as done and add a short summary

    Args:
        task_id: The id of the todo tak to be done
        result: A short summary of the task completed
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "Done"}

In [6]:
board_agent = Agent(
    MODEL, system_prompt="You help manage a shared todo baord", tools=[show_todos, plan_steps, complete_task]
)

In [7]:
result = await board_agent.run("What is on the board right now and what is the status of each?")
console.print(Markdown(result.output))

Here’s what’s on the board right now:                                                                              

 • Goal #1 (pending) — “Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.      
   Overwrite the file if it already exists.”                                                                       
    • No steps are listed under this goal yet.

### Using MCP


In [8]:
workspace = Path("workspace").resolve()

filesystem = MCPToolset(
    StdioTransport(command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)]),
    init_timeout=60,
)

In [9]:
file_agent = Agent(
    MODEL,
    instructions="You can read and write files within your workspace. Use the tools to perform the tasks needed",
    toolsets=[filesystem],
)

In [10]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it.")

console.print(Markdown(result.output))

The notes say that you’re welcoming the team and working together to build a small language tutor step by step. It 
instructs helpers to pick a single task from a shared board, complete it carefully, mark it as done, and to read   
the task closely and take your time.

### Agent in a loop


In [11]:
INSTRUCTIONS = (
    "You maintain a todo board."
    "When asked to plan, call show_todos to inspect the board. "
    "For each pending goal without steps, break it into an ordered "
    "checklist and call plan_steps to save it. "
    "Do not duplicate existing steps. "
    "Only mark tasks complete after the work has actually been performed."
)

worker = Agent(MODEL, instructions=INSTRUCTIONS, tools=[show_todos, plan_steps, complete_task], toolsets=[filesystem])

board.reset_board()
goal_id = board.add_goal(
    "Read notes.txt, translate its contents into Bengali and write it to bengali.txt."
    "Please overwrite if the file already exists"
)
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work with the pending goal on baord.")

print(result.output)
board.show_board()

Done — I read `notes.txt`, translated it into Bengali, and wrote the result to `bengali.txt` (overwriting/creating it as needed).


Goal #1: Read notes.txt, translate its contents into Bengali and write it to bengali.txt.Please overwrite if the file already exists  Translated notes.txt into Bengali and wrote the result to bengali.txt (overwrote/created file).